# SAE Scoping Mini-project — End-to-End Experiment

This notebook walks through **steps 1–7** of the SAE scoping experiment as described in the README.  
Heavy-compute steps (firing-rate calculation, model training) delegate to the scripts in `scripts/`  
via `!python` shell calls so you never reinvent the wheel.

| Step | What happens | How |
|------|-------------|-----|
| 1 | Fork & compute setup | Manual |
| 2 | Verify code | Python imports |
| 3 | Pick & preview dataset | Inline Python |
| 4 | Calculate & plot firing rates | `!python scripts/find_firing_rates.py` + `plot_firing_rates.py` |
| 5 | Evaluate SAE-hooked model at varying K | Inline Python |
| 6 | Train the LLM (recovery training) | `!python scripts/train_with_firing_rates.py` |
| 7 | Evaluate fine-tuned model in-domain & OOD | Inline Python |


## Setup
Run once to install all dependencies (idempotent).


In [ ]:
# Install the package and all dependencies
%pip install -e . -q
# Optionally log in to WandB (needed for training in step 6)
# import wandb; wandb.login()


## Step 1 — Fork & Compute Setup

Before running this notebook you should have:
1. **Forked** the repository to your own GitHub account so your results are tracked.
2. **Provisioned a GPU** (A100/H100 recommended; at minimum ~24 GB VRAM for Gemma-2-9B in bfloat16).  
   Runpod, Lambda Labs, or any cloud provider works fine.
3. Set the `WANDB_API_KEY` environment variable (or called `wandb.login()` above).
4. (Optional) Logged in to Hugging Face: `huggingface-cli login` if you hit rate-limits on gated models.

```bash
git clone <your-fork-url>
cd SAEScopingMiniproject
pip install -e .
```


## Step 2 — Verify the Code Works
Quick smoke-test: import the key library modules to make sure the install is clean.


In [ ]:
import torch
import sae_lens
from sae_scoping.trainers.sae_enhanced.rank import rank_neurons
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.trainers.sae_enhanced.train import train_sae_enhanced_model
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

print('torch:', torch.__version__)
print('sae_lens:', sae_lens.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Step 3 — Dataset Selection

**In-domain task:** Chemistry Q&A from [`4gate/StemQAMixture`](https://huggingface.co/datasets/4gate/StemQAMixture).

- Chosen because it is a well-defined scientific domain with clear boundaries.
- This is *not* biology, so our results are additive to any prior biology experiments.
- **Train/test split:** 90/10 random split seeded at 42 (no overlap).

**OOD tasks** (used in steps 5 & 7):
- **`codeparrot/apps`** — competitive programming (coding capability).
- **`HuggingFaceH4/ultrachat_200k`** — diverse general-purpose chat.

These two OOD tasks are chosen because they represent very different capability axes  
(reasoning/coding vs. social/general knowledge) and are clearly far from chemistry.


In [ ]:
from datasets import load_dataset
import pandas as pd

chem = load_dataset('4gate/StemQAMixture', 'chemistry', split='train')
print(f'Chemistry dataset: {len(chem):,} rows')
print('Columns:', chem.column_names)
# Preview a few rows
pd.DataFrame(chem.select(range(3)).to_dict()).T


In [ ]:
# Show train/test split sizes
splits = chem.train_test_split(test_size=0.1, seed=42)
print(f'Train : {len(splits["train"]):,} samples')
print(f'Test  : {len(splits["test"]):,} samples')


## Step 4 — Calculate & Plot Firing Rates

We run `scripts/find_firing_rates.py` which:
1. Loads the **three Gemmascope SAEs** for Gemma-2-9B-IT (layers 9, 20, 31 × width 16k).
2. For each `(dataset, SAE)` pair, counts how often each SAE latent fires on a token.
3. Saves a `distribution.safetensors` file under `scripts/.cache/`.

Then `scripts/plot_firing_rates.py` generates three sets of plots:
- **Sorted firing rates** — useful for picking a pruning threshold.
- **Cumulative firing rates** — shows how many neurons capture X% of activity.
- **Cross-dataset overlap** — which neurons are domain-specific vs. generic.

> ⏱ **Expected runtime:** ~30–90 min on a single A100 depending on dataset sizes.
> The script skips already-computed results, so it is safe to re-run.


In [ ]:
# Calculate firing rates for chemistry (in-domain) and two OOD datasets.
# -d  : comma-separated dataset names (stemqa_chemistry, apps, ultrachat)
# -i  : ignore padding tokens (True only)
# -b  : batch size
!python scripts/find_firing_rates.py -d stemqa_chemistry,apps,ultrachat -i True -b 4


In [ ]:
# Plot the saved distributions.
# -i : which ignore_padding setting to visualise
!python scripts/plot_firing_rates.py --ignore-padding True


In [ ]:
# Display the generated plots inline.
from pathlib import Path
from IPython.display import Image, display

plot_dir = Path('scripts/.cache/plots')
pngs = sorted(plot_dir.glob('*.png')) if plot_dir.exists() else []
if not pngs:
    print('No plots found — run the cells above first.')
else:
    for png in pngs:
        print(png.name)
        display(Image(filename=str(png)))


## Step 5 — Pick a Pruning Threshold and Evaluate the SAE-hooked Model

We sweep over several firing-rate thresholds `T` and measure **per-token cross-entropy loss**  
on the chemistry *test* set with the Gemmascope SAE pruned to only the neurons with  
`firing_rate ≥ T`.  The goal is to find the **lowest K** (fewest neurons) where in-domain loss  
is still close to the unpruned baseline — this K is what we hand to the training step.

> 💡 We use **layer 31** (the deepest available SAE) because it captures the most semantic
> information; you can repeat the sweep for layers 9 and 20 if you want.

> ⏱ Loading Gemma-2-9B takes ~2–3 min on a single GPU. The sweep itself depends on
> the number of samples (`N_EVAL_SAMPLES` below).


In [ ]:
from pathlib import Path
from safetensors.torch import load_file
import torch

CACHE_ROOT = Path('scripts/.cache/ignore_padding_True')
SAE_FOLDER = 'layer_31--width_16k--canonical'
DIST_PATH = CACHE_ROOT / 'stemqa_chemistry' / SAE_FOLDER / 'distribution.safetensors'

if not DIST_PATH.exists():
    print(f'Distribution file not found at {DIST_PATH}.  Run Step 4 first.')
else:
    dist = load_file(DIST_PATH)['distribution']
    print(f'SAE latent dimension : {len(dist):,}')
    print(f'Mean firing rate     : {dist.mean():.4f}')
    print()
    print(f'{'Threshold':>12}  {'Neurons kept':>14}  {'% of SAE':>10}')
    print('-' * 42)
    for t in [1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5]:
        n = int((dist >= t).sum().item())
        print(f'{t:>12.1e}  {n:>14,}  {100*n/len(dist):>9.2f}%')


In [ ]:
# ── Sweep: loss vs. K ────────────────────────────────────────────────────────
#
# This cell loads Gemma-2-9B once and evaluates cross-entropy loss at each
# threshold.  It uses the context-manager helpers from sae_scoping so no
# inference code needs to be duplicated here.
#
# Adjust N_EVAL_SAMPLES to trade speed for accuracy.

import gc
from contextlib import nullcontext
from functools import partial

import torch
from datasets import load_dataset as _lds
from sae_lens import SAE
from safetensors.torch import load_file
from transformers import AutoTokenizer, Gemma2ForCausalLM

from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_NAME    = 'google/gemma-2-9b-it'
SAE_RELEASE   = 'gemma-scope-9b-pt-res-canonical'
SAE_ID        = 'layer_31/width_16k/canonical'
HOOKPOINT     = 'model.layers.31'
DIST_PATH     = Path('scripts/.cache/ignore_padding_True/stemqa_chemistry/layer_31--width_16k--canonical/distribution.safetensors')
N_EVAL_SAMPLES = 64   # increase for a more accurate curve
BATCH_SIZE    = 4
MAX_LENGTH    = 512
# Thresholds to evaluate (plus None = no SAE / baseline)
THRESHOLDS    = [None, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load eval data (chemistry questions only, no answers needed for perplexity)
chem_raw = _lds('4gate/StemQAMixture', 'chemistry', split='train')
chem_test = chem_raw.train_test_split(test_size=0.1, seed=42)['test']
texts = chem_test.select(range(min(N_EVAL_SAMPLES, len(chem_test))))['question']

# ── Load tokenizer & model ───────────────────────────────────────────────────
print('Loading tokenizer and model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cpu', attn_implementation='eager'
)
model = model.to(device)
model.eval()

# ── Load distribution ────────────────────────────────────────────────────────
if not DIST_PATH.exists():
    raise FileNotFoundError(f'{DIST_PATH} — run Step 4 first')
dist = load_file(DIST_PATH)['distribution'].to(device)
neuron_ranking = torch.argsort(dist, descending=True)

# ── Load SAE once (pruning only masks activations, no weight copy needed) ───
print('Loading SAE...')
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)


def _eval_loss(pruned_sae=None):
    """Average per-token cross-entropy loss over `texts`."""
    total_loss, total_batches = 0.0, 0
    ctx = nullcontext()
    if pruned_sae is not None:
        hook_dict = {HOOKPOINT: partial(filter_hook_fn, SAEWrapper(pruned_sae))}
        ctx = named_forward_hooks(model, hook_dict)
    with torch.no_grad(), ctx:
        for i in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[i : i + BATCH_SIZE]
            enc = tokenizer(
                batch_texts, return_tensors='pt', padding=True,
                truncation=True, max_length=MAX_LENGTH
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc, labels=enc['input_ids'])
            total_loss += out.loss.item()
            total_batches += 1
    return total_loss / total_batches


# ── Sweep ────────────────────────────────────────────────────────────────────
results = []   # list of (threshold_label, n_neurons, loss)
for T in THRESHOLDS:
    if T is None:
        label, n_kept, pruned = 'baseline (no SAE)', len(dist), None
    else:
        n_kept = int((dist >= T).sum().item())
        pruned = get_pruned_sae(sae, neuron_ranking, K_or_p=n_kept, T=0.0).to(device)
        label = f'T={T:.0e}  (K={n_kept:,})'
    loss = _eval_loss(pruned)
    results.append((label, n_kept, loss))
    print(f'{label:40s}  loss={loss:.4f}')
    if pruned is not None:
        del pruned
        gc.collect()
        torch.cuda.empty_cache()

print('\nDone.')


In [ ]:
import matplotlib.pyplot as plt

# Plot loss vs. number of neurons kept
labels = [r[0] for r in results]
ns     = [r[1] for r in results]
losses = [r[2] for r in results]
baseline_loss = losses[0]  # no-SAE reference

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns[1:], losses[1:], marker='o', label='Pruned SAE')
ax.axhline(baseline_loss, color='gray', linestyle='--', label=f'Baseline (no SAE): {baseline_loss:.4f}')
ax.set_xscale('log')
ax.set_xlabel('Number of neurons kept (K)')
ax.set_ylabel('Avg cross-entropy loss (chemistry test)')
ax.set_title('In-domain loss vs. pruning aggressiveness')
ax.legend()
fig.tight_layout()
plt.savefig('scripts/.cache/plots/loss_vs_K.png', dpi=150)
plt.show()

# Recommend a threshold: last T where loss is within 5% of baseline
tolerance = 0.05
chosen_T, chosen_K = None, None
for T, (_, n, loss) in zip(THRESHOLDS[1:], results[1:]):
    if (loss - baseline_loss) / baseline_loss <= tolerance:
        chosen_T, chosen_K = T, n
if chosen_T:
    print(f'\n✅ Recommended threshold : T = {chosen_T:.0e}  (K = {chosen_K:,} neurons)')
    print(f'   This keeps loss within {tolerance*100:.0f}% of the no-SAE baseline.')
else:
    print('All thresholds degrade loss by >5%.  Consider using a lower threshold or more neurons.')


## Step 6 — Train the LLM (Recovery Training)

We call `scripts/train_with_firing_rates.py` which:
1. Loads the distribution file identified in Step 5.
2. Prunes the SAE to the `K` neurons selected above.
3. Freezes all model layers **before** the SAE hookpoint and trains the layers **after** it.
4. Logs metrics to Weights & Biases.
5. Saves checkpoints to `./outputs_gemma9b/`.

**Edit the variables below** to match your chosen threshold and W&B project.

> ⏱ 1 000 gradient-update steps with batch=2, accum=8 takes ~2–4 hours on a single A100.
> Start with `--max-steps 100` to confirm the pipeline runs end-to-end before committing.


In [ ]:
# ── Configure these variables before running ─────────────────────────────────
CHOSEN_THRESHOLD  = 1e-4    # from Step 5 sweep  (e.g. 1e-4)
WANDB_PROJECT     = 'gemma-scope-9b-chemistry-scoping'
MAX_STEPS         = 1000    # increase for a full run
BATCH_SIZE_TRAIN  = 2
ACCUM_STEPS       = 8       # effective batch = BATCH_SIZE_TRAIN * ACCUM_STEPS = 16
DIST_PATH_STR     = 'scripts/.cache/ignore_padding_True/stemqa_chemistry/layer_31--width_16k--canonical/distribution.safetensors'

# Build the command
cmd = (
    f'python scripts/train_with_firing_rates.py'
    f' --dist-path "{DIST_PATH_STR}"'
    f' --train-on-dataset chemistry'
    f' --threshold {CHOSEN_THRESHOLD}'
    f' --batch-size {BATCH_SIZE_TRAIN}'
    f' --accum {ACCUM_STEPS}'
    f' --max-steps {MAX_STEPS}'
    f' --wandb-project-name "{WANDB_PROJECT}"'
    f' --save-every 500'
    f' --save-limit 5'
)
print('Command to run:\n')
print(cmd)


In [ ]:
# Run the training script.
# If you see OOM errors, reduce --batch-size to 1 and increase --accum to 16.
import subprocess, shlex
!{cmd}


In [ ]:
# After training, list the saved checkpoints
import os
from pathlib import Path

checkpoint_dirs = sorted(Path('./outputs_gemma9b').rglob('checkpoint-*'))
if not checkpoint_dirs:
    print('No checkpoints found yet — training may still be running.')
else:
    print('Available checkpoints:')
    for d in checkpoint_dirs:
        print(' ', d)
    CHECKPOINT_PATH = str(checkpoint_dirs[-1])  # pick the latest
    print(f'\nUsing checkpoint: {CHECKPOINT_PATH}')


## Step 7 — Evaluate the Fine-tuned SAE-hooked Model

We load the checkpoint saved in Step 6 and evaluate cross-entropy loss on:
- **Chemistry (in-domain)** — should be ≈ as good as the original model.
- **Apps (OOD — coding)** — should be *worse* than the original model.
- **Ultrachat (OOD — general chat)** — should be *worse* than the original model.

If scoping worked, the fine-tuned model should be good in-domain and bad OOD.  
Compare each loss against the original Gemma-2-9B-IT baseline (no SAE, no fine-tuning).


In [ ]:
# ── Config — set CHECKPOINT_PATH if the previous cell did not set it ──────────
# CHECKPOINT_PATH = './outputs_gemma9b/chemistry/layer_31_width_16k_canonical_h0.0001_<hash>/checkpoint-1000'

import gc
from contextlib import nullcontext
from functools import partial
from pathlib import Path

import torch
from datasets import load_dataset as _lds
from sae_lens import SAE
from safetensors.torch import load_file
from transformers import AutoTokenizer, Gemma2ForCausalLM

from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

MODEL_NAME  = 'google/gemma-2-9b-it'
SAE_RELEASE = 'gemma-scope-9b-pt-res-canonical'
SAE_ID      = 'layer_31/width_16k/canonical'
HOOKPOINT   = 'model.layers.31'
DIST_PATH   = Path('scripts/.cache/ignore_padding_True/stemqa_chemistry/layer_31--width_16k--canonical/distribution.safetensors')
N_EVAL      = 64
BATCH_SIZE  = 4
MAX_LENGTH  = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Load evaluation texts ────────────────────────────────────────────────────
chem_raw  = _lds('4gate/StemQAMixture', 'chemistry', split='train')
chem_test = chem_raw.train_test_split(test_size=0.1, seed=42)['test']
apps_raw  = _lds('codeparrot/apps', split='train')
uc_raw    = _lds('HuggingFaceH4/ultrachat_200k', split='test_sft')

eval_texts = {
    'chemistry (in-domain)' : chem_test.select(range(min(N_EVAL, len(chem_test))))['question'],
    'apps (OOD — coding)'   : apps_raw.select(range(min(N_EVAL, len(apps_raw))))['question'],
    'ultrachat (OOD — chat)': [
        msgs[0]['content'] for msgs in
        uc_raw.select(range(min(N_EVAL, len(uc_raw))))['messages']
    ],
}


def _eval_loss(model, texts, pruned_sae=None):
    total, count = 0.0, 0
    ctx = nullcontext()
    if pruned_sae is not None:
        ctx = named_forward_hooks(model, {HOOKPOINT: partial(filter_hook_fn, SAEWrapper(pruned_sae))})
    with torch.no_grad(), ctx:
        for i in range(0, len(texts), BATCH_SIZE):
            enc = tokenizer(
                list(texts[i : i + BATCH_SIZE]), return_tensors='pt',
                padding=True, truncation=True, max_length=MAX_LENGTH
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc, labels=enc['input_ids'])
            total += out.loss.item()
            count += 1
    return total / count


# ── Load distribution and pruned SAE ─────────────────────────────────────────
dist           = load_file(DIST_PATH)['distribution'].to(device)
neuron_ranking = torch.argsort(dist, descending=True)
n_kept         = int((dist >= CHOSEN_THRESHOLD).sum().item())
print(f'Keeping {n_kept:,}/{len(dist):,} neurons at threshold {CHOSEN_THRESHOLD:.0e}')

base_sae    = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
pruned_sae  = get_pruned_sae(base_sae, neuron_ranking, K_or_p=n_kept, T=0.0).to(device)

# ── 7a. Evaluate BASELINE model (original Gemma-2-9B-IT, no fine-tuning) ─────
print('\nLoading baseline model (original Gemma-2-9B-IT)...')
baseline_model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cpu', attn_implementation='eager'
).to(device)
baseline_model.eval()

baseline_results = {}
for name, texts in eval_texts.items():
    loss = _eval_loss(baseline_model, texts)
    baseline_results[name] = loss
    print(f'  Baseline  |  {name:30s}  |  loss = {loss:.4f}')

del baseline_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ── 7b. Evaluate FINE-TUNED + PRUNED-SAE model ───────────────────────────────
print(f'\nLoading fine-tuned model from {CHECKPOINT_PATH}...')
ft_model = Gemma2ForCausalLM.from_pretrained(
    CHECKPOINT_PATH, torch_dtype=torch.bfloat16, device_map='cpu', attn_implementation='eager'
).to(device)
ft_model.eval()

ft_results = {}
for name, texts in eval_texts.items():
    loss = _eval_loss(ft_model, texts, pruned_sae=pruned_sae)
    ft_results[name] = loss
    print(f'  Fine-tuned|  {name:30s}  |  loss = {loss:.4f}')


In [ ]:
# ── 7c. Summary plot ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

datasets  = list(baseline_results.keys())
base_vals = [baseline_results[d] for d in datasets]
ft_vals   = [ft_results[d]       for d in datasets]

x   = np.arange(len(datasets))
w   = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, base_vals, w, label='Baseline (original Gemma-2-9B-IT)')
ax.bar(x + w/2, ft_vals,   w, label=f'Fine-tuned + pruned SAE (K={n_kept:,})')
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=15, ha='right')
ax.set_ylabel('Avg cross-entropy loss (lower = better)')
ax.set_title('Scoping evaluation: in-domain vs. OOD')
ax.legend()
fig.tight_layout()
plt.savefig('scripts/.cache/plots/scoping_eval.png', dpi=150)
plt.show()

print('\nInterpretation:')
for d in datasets:
    delta = ft_results[d] - baseline_results[d]
    direction = 'BETTER' if delta < 0 else 'WORSE'
    print(f'  {d:35s}: Δloss = {delta:+.4f}  ({direction} after scoping)')
print('\nFor successful scoping: in-domain should be BETTER (or neutral), OOD should be WORSE.')
